# LOGAN Neural Fraïssé Quickstart

Run one God-vs-Devil fight and a small held-out benchmark. Active symbolic Devil, learned Builder. Not a GAN claim, not general model theory — see the caveats at the end.

**In Colab:** Runtime → Run all. The first cell clones and installs LOGAN.

In [ ]:
# Colab bootstrap: clone + install LOGAN when running in Google Colab.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/Mircus/Logan.git"
BRANCH = "master"
REPO_DIR = "/content/Logan"

if "google.colab" in sys.modules:
    if not pathlib.Path(REPO_DIR).exists():
        subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])
else:
    print("Not running in Colab; assuming a local editable install or repo root.")

## What this notebook runs
1. Shows the example problem `cycle3_fight.json`.
2. Runs the fight backend and prints the alternating Devil/God/Judge trace.
3. Shows the final structure and witness.
4. Runs a small held-out benchmark (train n=3,4; test n=5; 20 tasks).

In [ ]:
import json, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT if (ROOT / 'src').exists() else ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
PROBLEM = ROOT / 'examples' / 'problems' / 'cycle3_fight.json'
print(PROBLEM.read_text())

In [ ]:
from logical_gans.modelbuilder.neural_fraisse.fight import load_problem, run_fight, render
problem = load_problem(PROBLEM)
result = run_fight(problem, 'active_symbolic')
print(render(result, problem))

In [ ]:
# first 8 trace events
for ev in result['events'][:8]:
    print(ev.actor, ev.payload)

In [ ]:
# final structure and witness
print('outcome:', result['outcome'])
print('structure:', json.dumps(result['structure'], indent=2))
print('witness:', json.dumps(result['witness'], indent=2))

In [ ]:
# small held-out benchmark (train n=3,4 -> test n=5)
from logical_gans.modelbuilder.neural_fraisse.benchmark import run_benchmark
from logical_gans.modelbuilder.neural_fraisse import results as R
res = run_benchmark([3, 4], [5], n_tasks=20, budget=800, seed=0, epochs=120, verbose=False)
print(R.format_table(res['aggregate']))
print('verdict:', res['verdict'])

## Caveats
- One controlled cyclic task family.
- Active symbolic Devil, not learned Devil.
- Learned Builder only.
- Uses generic game-context features, including collision/image features.
- Not a GAN claim.
- Not a general model-theory claim.